In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on sys.path so we can import from src/
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os
import numpy as np
import pandas as pd
import wandb
import torch
from torch import optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets
from tqdm import tqdm

from fractal_sweep_config import sweep_config
from src.tools import get_transforms

# Enable cuDNN benchmark for faster convolutions on fixed input resolutions
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

In [ ]:
# importing local modules
from src.fractal_functions import (alpha_fractalize, alpha_fractalize_first_derivative)
from src.fractal_activation_N import FractalActivationN

# Base functions for fractalization: x, x², x³
# (only the positive/nonzero part — negatives are handled by FractalActivationN which outputs 0 for x < 0)
def identity(x):
    return np.asarray(x, dtype=float)

def d_identity(x):
    return np.ones_like(np.asarray(x, dtype=float))

# Base functions for squared and cubic ReLU (uncomment when needed)
# def square(x):
#     return np.asarray(x, dtype=float) ** 2

# def d_square(x):
#     return 2.0 * np.asarray(x, dtype=float)

# def cube(x):
#     return np.asarray(x, dtype=float) ** 3

# def d_cube(x):
#     return 3.0 * np.asarray(x, dtype=float) ** 2

In [ ]:
# Fractal precomputation parameters (nonzero-part-only on [0, 1])
a = 0       # Fractal domain starts at 0 (negatives → 0)
b = 1       # Fractal domain ends at b (above b → classical)
n_subintervals = 2
n_iter = 2

# Perturbation: vanishes at x=0 and x=b so boundary conditions match
def _perturbation(x):
    x = np.asarray(x, dtype=float)
    return x**2 * (x - b)**2

def _d_perturbation(x):
    x = np.asarray(x, dtype=float)
    return 2.0 * x * (x - b) * (2.0 * x - b)

def g_identity(x):
    return identity(x) + _perturbation(x)

def dg_identity(x):
    return d_identity(x) + _d_perturbation(x)

# Perturbation for square and cube (uncomment when needed)
# def g_square(x):
#     return square(x) + _perturbation(x)

# def dg_square(x):
#     return d_square(x) + _d_perturbation(x)

# def g_cube(x):
#     return cube(x) + _perturbation(x)

# def dg_cube(x):
#     return d_cube(x) + _d_perturbation(x)

# Dictionaries to store precomputed fractal activation LUTs
relu_fractals = {}
d_relu_fractals = {}

In [ ]:
class CNNModel(nn.Module):
    def __init__(self,
                 filters,                    # List of filters => Controls number of conv layers and filter sizes
                 kernel_size,                # Size of filters
                 activation,                 # Activation function
                 dropout,                    # Dropout rate (optional)
                 use_batchnorm,              # Whether to use batch norm (optional)
                 alpha1=0.2,                 # Alpha 1 for fractalization
                 alpha2=0.2,                 # Alpha 2 for fractalization
                 input_shape=(3, 192, 192),  # Input shape compatible with iNaturalist dataset (192x192)
                 dense_units=256,            # Number of neurons in the dense (fully connected) layer
                 num_classes=10):            # Output layer with 10 neurons

        super().__init__()
        
        # Retrieve precomputed fractal LUT or compute if needed
        alpha = [alpha1, alpha2]
        if (alpha1, alpha2) in relu_fractals:
            identity_fractal = relu_fractals[(alpha1, alpha2)]
        else:
            identity_fractal = alpha_fractalize(identity, g_identity, a, b, n_subintervals, alpha, n_iter, True)

        # Precompute square and cube fractals if needed (uncomment when using squared/cubic relu)
        # square_fractal = alpha_fractalize(square, g_square, a, b, n_subintervals, alpha, n_iter, True)
        # cube_fractal = alpha_fractalize(cube, g_cube, a, b, n_subintervals, alpha, n_iter, True)

        def get_activation(name):
            name = name.lower()
            if name == "f_relu":
                return FractalActivationN(identity_fractal, lambda x: x)
            # if name == "f_squared_relu":
            #     return FractalActivationN(square_fractal, lambda x: x ** 2)
            # if name == "f_cubic_relu":
            #     return FractalActivationN(cube_fractal, lambda x: x ** 3)
            raise ValueError(f"Unsupported activation: {name}")

        layers = []
        in_channels = input_shape[0]

        # Building conv-activation-maxpool blocks
        for out_channels in filters:
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=1))  # Conv layer
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_channels))   # Optional BatchNorm
            layers.append(get_activation(activation))         # Fractal activation module
            layers.append(nn.MaxPool2d(2))                    # Max pooling
            if dropout > 0:
                layers.append(nn.Dropout(dropout))            # Optional dropout
            in_channels = out_channels

        # Feature extractor with conv-activation-maxpool blocks
        self.features = nn.Sequential(*layers)

        # Automatically calculate the output size after conv layers for the FC layer
        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            out = self.features(dummy)
            flatten_size = out.view(1, -1).shape[1]

        # Classifier block with dense + activation + dropout + output
        self.classifier = nn.Sequential(
            nn.Linear(flatten_size, dense_units),   # First dense layer
            get_activation(activation),             # Fractal activation in dense layer
            nn.Dropout(dropout),                    # Dropout
            nn.Linear(dense_units, num_classes)     # Output layer with 10 neurons
        )

    def forward(self, x):
        x = self.features(x)            # Pass through convolutional blocks
        x = torch.flatten(x, 1)         # Flatten before fully connected layers
        return self.classifier(x)       # Output logits for classification

In [ ]:
wandb.login(key="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")

In [ ]:
# Paths for global best accuracy and checkpoint across ALL (alpha1, alpha2) combinations
GLOBAL_BEST_PATH = (PROJECT_ROOT / "fractal_N" / "best_accuracy.txt").resolve()
GLOBAL_BEST_MODEL_PATH = (PROJECT_ROOT / "fractal_N" / "best_model.pth").resolve()
GLOBAL_BEST_VAL_ACC = 0.0

# If a previous global best file exists, load it so we NEVER reset across combinations or notebook runs
if GLOBAL_BEST_PATH.exists():
    try:
        with open(GLOBAL_BEST_PATH, "r") as f:
            line = f.readline().strip()
            if ":" in line:
                GLOBAL_BEST_VAL_ACC = float(line.split(":")[1].strip())
            else:
                GLOBAL_BEST_VAL_ACC = float(line)
        print(f"Loaded existing global best val_acc across all combinations: {GLOBAL_BEST_VAL_ACC:.4f}")
    except Exception:
        GLOBAL_BEST_VAL_ACC = 0.0

# Training function
def train():
    global GLOBAL_BEST_VAL_ACC

    # Initialize wandb
    wandb.init()
    config = wandb.config

    # Use the current combination of alpha1 and alpha2 from loop (or config if provided)
    alpha1_val = getattr(config, "alpha1", current_alpha1 if "current_alpha1" in globals() else 0.2)
    alpha2_val = getattr(config, "alpha2", current_alpha2 if "current_alpha2" in globals() else 0.2)
    dense_units_val = getattr(config, "dense_units", 256)

    # Update wandb config with alpha1 and alpha2
    wandb.config.update({"alpha1": alpha1_val, "alpha2": alpha2_val}, allow_val_change=True)

    # Generating a meaningful run name using config values including alpha1 and alpha2
    run_name = f"run_a1-{alpha1_val}_a2-{alpha2_val}_filters-{config.filters_per_layer}_act-{config.activation}_bs-{config.batch_size}_lr-{config.learning_rate}_do-{config.dropout_rate}_bn-{config.use_batchnorm}_aug-{config.augmentation}"
    wandb.run.name = run_name

    # Transforms
    train_tf, val_tf = get_transforms(config.augmentation)

    # Loading datasets
    data_root = PROJECT_ROOT / "inaturalist_12K"
    train_data = datasets.ImageFolder(str(data_root / "train"), transform=train_tf)
    val_data = datasets.ImageFolder(str(data_root / "val"), transform=val_tf)

    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    num_workers = 2

    train_loader = DataLoader(
        train_data,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=use_cuda,
        persistent_workers=(num_workers > 0)
    )
    val_loader = DataLoader(
        val_data,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=use_cuda,
        persistent_workers=(num_workers > 0)
    )

    # Preparing model
    filters = config.filters_per_layer
    model = CNNModel(
        filters=filters,
        kernel_size=3,
        activation=config.activation,
        dropout=config.dropout_rate,
        use_batchnorm=config.use_batchnorm,
        alpha1=alpha1_val,
        alpha2=alpha2_val,
        input_shape=(3, 192, 192),
        dense_units=dense_units_val
    )
    model.to(device)

    # Loss & optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

    # Automatic Mixed Precision (AMP) scaler
    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)

    # Training loop
    for epoch in range(config.epochs):
        print(f"\nEpoch {epoch + 1}/{config.epochs}")
        print("-" * 60)
        model.train()
        total_loss, correct, total = 0, 0, 0

        for inputs, labels in tqdm(train_loader, desc="Training Progress", ncols=100, colour="magenta"):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=use_cuda):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_acc = correct / total

        # Validation loop
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validation Progress", ncols=100, colour="cyan"):
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with torch.amp.autocast("cuda", enabled=use_cuda):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")
        print("-" * 60)

        wandb.log({
            "epoch": epoch + 1,
            "alpha1": alpha1_val,
            "alpha2": alpha2_val,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

    # Synchronize with file on disk in case updated externally
    if GLOBAL_BEST_PATH.exists():
        try:
            with open(GLOBAL_BEST_PATH, "r") as f:
                line = f.readline().strip()
                file_best = float(line.split(":")[1].strip()) if ":" in line else float(line)
                GLOBAL_BEST_VAL_ACC = max(GLOBAL_BEST_VAL_ACC, file_best)
        except Exception:
            pass

    # Save model and update best_accuracy.txt ONLY if it strictly beats the overall global best
    if val_acc > GLOBAL_BEST_VAL_ACC:
        GLOBAL_BEST_VAL_ACC = val_acc
        torch.save(model.state_dict(), str(GLOBAL_BEST_MODEL_PATH))
        with open(GLOBAL_BEST_PATH, "w") as f:
            f.write(f"val_acc: {val_acc:.4f}\n")
            f.write(f"alpha1: {alpha1_val}\n")
            f.write(f"alpha2: {alpha2_val}\n")
            f.write(f"filters_per_layer: {config.filters_per_layer}\n")
            f.write(f"activation: {config.activation}\n")
            f.write(f"dense_units: {dense_units_val}\n")
            f.write(f"learning_rate: {config.learning_rate}\n")
            f.write(f"batch_size: {config.batch_size}\n")
            f.write(f"dropout_rate: {config.dropout_rate}\n")
            f.write(f"use_batchnorm: {config.use_batchnorm}\n")
            f.write(f"augmentation: {config.augmentation}\n")
            f.write(f"epochs: {config.epochs}\n")
        print("\n" + "*" * 60)
        print("*** NEW OVERALL BEST MODEL ACROSS ALL COMBINATIONS! ***")
        print(f"val_acc: {val_acc:.4f} ({val_acc*100:.2f}%) | alpha1: {alpha1_val}, alpha2: {alpha2_val}")
        print(f"Saved to: {GLOBAL_BEST_PATH}")
        print("*" * 60 + "\n")
    else:
        print(f"Run val_acc: {val_acc:.4f} did not beat overall global best: {GLOBAL_BEST_VAL_ACC:.4f} (best_accuracy.txt retained)")

    wandb.finish()
    print("Training run complete.")

In [ ]:
ALPHA1 = [0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45] 
ALPHA2 = [0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]

# Number of sweep runs per alpha combination (adjust count as desired, e.g. 2 or None for full grid)
SWEEP_RUNS_PER_COMBO = 2

for alpha1 in ALPHA1:
    for alpha2 in ALPHA2:
        alpha = [alpha1, alpha2]
        relu_fractal = alpha_fractalize(identity, g_identity, a, b, n_subintervals, alpha, n_iter, True)
        d_relu_fractal = alpha_fractalize_first_derivative(d_identity, dg_identity, a, b, n_subintervals, alpha, n_iter, True)
        relu_fractals[(alpha1, alpha2)] = relu_fractal
        d_relu_fractals[(alpha1, alpha2)] = d_relu_fractal

        # Set current combination of alpha1 and alpha2 for train()
        current_alpha1 = alpha1
        current_alpha2 = alpha2

        print("\n" + "=" * 70)
        print(f"Running sweep for combination: alpha1 = {alpha1}, alpha2 = {alpha2}")
        print(f"Current overall best val_acc across all combinations so far: {GLOBAL_BEST_VAL_ACC*100:.2f}%")
        print("=" * 70)

        # Launch wandb sweep agent with train() for this (alpha1, alpha2) combination
        sweep_id = wandb.sweep(sweep_config, project="fractal_CNN")
        wandb.agent(sweep_id, function=train, count=SWEEP_RUNS_PER_COMBO)
        wandb.finish()

print("\n" + "=" * 70)
print("Optimization across ALL combinations of alpha1 and alpha2 complete!")
if GLOBAL_BEST_PATH.exists():
    print(f"\n--- Overall Global Best Result across all combinations ({GLOBAL_BEST_PATH}) ---")
    with open(GLOBAL_BEST_PATH, "r") as f:
        print(f.read())
print("=" * 70)